# Chandra OCR 2 분석 서버 (FastAPI)

이 노트북은 `datalab-to/chandra-ocr-2` 모델을 사용하여 이미지 분석 및 질의응답 API를 제공합니다.

In [ ]:
# !pip install fastapi uvicorn torch transformers pillow python-multipart python-dotenv

import uvicorn
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.middleware.cors import CORSMiddleware
import os
import shutil
from chandra.ocr_service import analyzeImageWithQuestion

app = FastAPI()

# CORS 설정: 모든 허용
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

UPLOAD_DIR = "chandra/dataset"
if not os.path.exists(UPLOAD_DIR):
    os.makedirs(UPLOAD_DIR)

@app.post("/analyze")
async def analyze(image: UploadFile = File(...), question: str = Form(...)):
    """ 이미지를 업로드받아 분석하고 질문에 답하는 API입니다. """
    try:
        filePath = os.path.join(UPLOAD_DIR, image.filename)
        
        # 파일 저장
        with open(filePath, "wb") as buffer:
            shutil.copyfileobj(image.file, buffer)
            
        # 분석 수행
        result = analyzeImageWithQuestion(filePath, question)
        
        # 결과 반환 (JSON)
        return result
        
    except Exception as e:
        return {"success": False, "message": str(e)}

if __name__ == "__main__":
    # 주피터 노트북에서 실행 시 포트 충돌 주의
    uvicorn.run(app, host="0.0.0.0", port=8000)